# M7.4 · Direct Preference Optimization (DPO) — production recipe, single GPU

Runs the **same container-based recipe as the production Nemotron SDG-RL DPO pipeline**
(`4.sdg_rl/dpo_run/run_dpo.sh`), scaled to **one** A100 / H100 / H200.

**Same as production:**
- Engine: **NeMo-RL** in **`nvcr.io/nvidia/nemo-rl:v0.6.0`**.
- Launch: `/opt/nemo_rl_venv/bin/python -u examples/run_dpo.py --config <recipe>.yaml`
  from `/opt/nemo-rl` (container runs as **root** — the NeMo-RL venv is not usable with `-u`).
- Data: NeMo-RL **PreferenceDataset** JSONL (`context` + ranked `completions`).
- Recipe pattern: YAML override inheriting the container base config via `defaults:`.

**Scaled for 1 GPU (the deltas):**
- Model: **`Qwen/Qwen2.5-0.5B`** (production: 30B MoE Nemotron from SFT `LOWEST_VAL`).
- `cluster.gpus_per_node: 1` (production: 8 on H100 NVL).
- `max_input_seq_length: 1024` (was 4096), `max_num_steps: 20` (was 2000).
- No DeepEP / expert-parallel — not needed on a single-GPU dense run.

**Flow:** pull container → write `recipes/dpo_1gpu.yaml` → `run_dpo.py` (in container) →
checkpoint in `work/dpo_checkpoints/`, consumed by M8 evaluation.

Training data: `data/dpo_train.jsonl` (18 records) and `data/dpo_val.jsonl` (4 records).


## 1. Prerequisites — GPU, NGC login, HF token, pull the container

All training runs **inside** the NeMo-RL container (same host-spawns-container pattern as
`run_dpo.sh`). The host only needs Docker + a GPU plus an NGC API key (pull) and an HF
token if your model is gated.

Split bind-mounts keep checkpoints on `/data` (`docker_storage.py`), matching M7.2 CPT
and M7.3 SFT.


In [1]:
import os, subprocess, getpass, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
from docker_storage import ensure_docker_storage, workshop_work_dir, docker_workspace_volumes
from notebook_env import bootstrap_notebook_env, ensure

ensure_docker_storage()
bootstrap_notebook_env()
ensure("torch", ["torch>=2.5.0,<2.7.0"], quiet=True)

import torch
assert torch.cuda.is_available(), "DPO needs a CUDA GPU (1x A100 / H100 / H200)."
_p = torch.cuda.get_device_properties(0)
print(f"GPU: {_p.name} ({_p.total_memory / 2**30:.1f} GiB)")

if not os.environ.get("NGC_API_KEY"):
    os.environ["NGC_API_KEY"] = getpass.getpass("NGC API key: ").strip()
if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass.getpass("HuggingFace token (if needed): ").strip() or ""
assert os.environ.get("NGC_API_KEY"), "NGC_API_KEY required to pull the container."

NB_DIR = Path.cwd().resolve()
WORK_DIR = workshop_work_dir("M7-model_training")
CONTAINER = "nvcr.io/nvidia/nemo-rl:v0.6.0"
BASE_MODEL = "Qwen/Qwen2.5-0.5B"   # workshop: small dense model for 1× A100
MODEL_PATH = BASE_MODEL            # production: /workspace/latest_sft_checkpoint

for p in (NB_DIR / "data" / "dpo_train.jsonl", NB_DIR / "data" / "dpo_val.jsonl"):
    assert p.exists(), f"Missing {p}"

print("Docker login -> nvcr.io ...")
subprocess.run(["docker", "login", "nvcr.io", "-u", "$oauthtoken", "--password-stdin"],
               input=os.environ["NGC_API_KEY"].encode(), check=True)
print("Pulling", CONTAINER, "(first time: several GB) ...")
subprocess.check_call(["docker", "pull", CONTAINER])

# Reusable docker-run prefix — mirrors run_dpo.sh host-side docker run.
# NeMo-RL v0.6.0: python lives in /opt/nemo_rl_venv (root-only). Run as root and
# bypass the NVIDIA entrypoint (it looks for `python` on PATH and fails with exit 127).
NEMO_PYTHON = "/opt/nemo_rl_venv/bin/python"
DOCKER = [
    "docker", "run", "--rm", "--gpus", "device=0",
    "--entrypoint", "",
    "--ipc=host", "--network=host",
    "--shm-size=16g", "--ulimit", "memlock=-1", "--ulimit", "stack=67108864",
    "-e", "HOME=/tmp",
    "-e", "HF_HOME=/workspace/work/hf_cache",
    "-e", "HF_HUB_ENABLE_HF_TRANSFER=1",
    "-e", "PYTHONUNBUFFERED=1",
    "-e", "PYTORCH_ALLOC_CONF=expandable_segments:True",
    *([] if not os.environ.get("HF_TOKEN") else ["-e", f"HF_TOKEN={os.environ['HF_TOKEN']}"]),
    *docker_workspace_volumes(NB_DIR, WORK_DIR),
]
print("workspace work dir (checkpoints):", WORK_DIR)
print("policy model:", MODEL_PATH)


2026-06-16 09:08:46,521 INFO === ensure_docker_storage (storage=/data, log: /data/logs/docker_storage.log) ===
2026-06-16 09:08:46,522 INFO disk /: 133.0G used / 983.1G (13.5%)
2026-06-16 09:08:46,522 INFO disk /data: 133.0G used / 983.1G (809.8G free)
2026-06-16 09:08:46,523 INFO env TMPDIR=/data/cache/tmp
2026-06-16 09:08:46,524 INFO env DOCKER_TMPDIR=/data/cache/tmp
2026-06-16 09:08:46,524 INFO env PIP_CACHE_DIR=/data/cache/pip
2026-06-16 09:08:46,525 INFO env UV_CACHE_DIR=/data/cache/uv
2026-06-16 09:08:46,525 INFO env HF_HOME=/data/cache/hf
2026-06-16 09:08:46,526 INFO env XDG_CACHE_HOME=/data/cache/xdg
2026-06-16 09:08:46,526 INFO env LOCAL_NIM_CACHE=/data/cache/nim
2026-06-16 09:08:46,527 INFO $ docker info --format {{.DockerRootDir}}
2026-06-16 09:08:46,580 INFO docker data-root (config): /data/docker
2026-06-16 09:08:46,581 INFO docker data-root (live):   /data/docker
2026-06-16 09:08:46,581 INFO $ docker info
2026-06-16 09:08:46,636 INFO $ docker info --format {{.DockerRootDi

docker storage ok: /data/docker (809.8G free on /data)
notebook env: /home/shadeform/workshop-materials-gsi/repo-content_v2/workshop-Materials/M7-model_training/.venv (python /home/shadeform/workshop-materials-gsi/repo-content_v2/workshop-Materials/M7-model_training/.venv/bin/python, storage /data)
ok: torch


/home/shadeform/workshop-materials-gsi/repo-content_v2/workshop-Materials/M7-model_training/.venv/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:275: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:81.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


GPU: NVIDIA A100-SXM4-80GB (79.2 GiB)


NGC API key:  ········
HuggingFace token (if needed):  ········


Docker login -> nvcr.io ...
Login Succeeded
Pulling nvcr.io/nvidia/nemo-rl:v0.6.0 (first time: several GB) ...
v0.6.0: Pulling from nvidia/nemo-rl
0622fac788ed: Pulling fs layer
4f4fb700ef54: Pulling fs layer
bd0ed3dadbe9: Pulling fs layer
7b57f70af223: Pulling fs layer
3f0b11d337e6: Pulling fs layer
2104594958ce: Pulling fs layer
ba15f2616882: Pulling fs layer
7b57f70af223: Waiting
4e46d4ab7302: Pulling fs layer
3f0b11d337e6: Waiting
50f087002df9: Pulling fs layer
2104594958ce: Waiting
ba15f2616882: Waiting
4e46d4ab7302: Waiting
f94296dbf484: Pulling fs layer
50f087002df9: Waiting
03a8530f6876: Pulling fs layer
f94296dbf484: Waiting
cce238fffcb6: Pulling fs layer
64a55035aee6: Pulling fs layer
1394c771d714: Pulling fs layer
03a8530f6876: Waiting
2312e005f291: Pulling fs layer
cce238fffcb6: Waiting
64a55035aee6: Waiting
1394c771d714: Waiting
95f88b748512: Pulling fs layer
2312e005f291: Waiting
b34261a35067: Pulling fs layer
b29ea1b3ef7d: Pulling fs layer
3663ad84e495: Pulling fs layer


## 2. Write the single-GPU DPO recipe

- `data.max_input_seq_length: 2048` — NeMo-RL **drops** examples whose tokenized
  prompt+completion exceeds this limit. With only 18 train records, an all-filtered
  batch causes `KeyError: global_valid_toks`; workshop data is kept short on purpose.


In [2]:
DPO_RECIPE = f"""# Single-GPU DPO recipe (workshop). Mirrors 4.sdg_rl/dpo_run production override.
# Inherits defaults from the container base config (production inherits the 30B MoE recipe).
defaults: /opt/nemo-rl/examples/configs/dpo.yaml

policy:
  model_name: {MODEL_PATH}
  tokenizer:
    name: {MODEL_PATH}
  train_global_batch_size: 2
  train_micro_batch_size: 1
  max_total_sequence_length: 2048
  dtensor_cfg:
    activation_checkpointing: true
    tensor_parallel_size: 1
    env_vars:
      PYTORCH_CUDA_ALLOC_CONF: expandable_segments:True
  automodel_kwargs:
    trust_remote_code: true
    force_hf: true

data:
  max_input_seq_length: 2048
  shuffle: true
  num_workers: 0
  train:
    data_path: /workspace/data/dpo_train.jsonl
    dataset_name: PreferenceDataset
  validation:
    data_path: /workspace/data/dpo_val.jsonl
    dataset_name: PreferenceDataset
  default:
    dataset_name: PreferenceDataset
    prompt_file: null
    system_prompt_file: null
  val_data_paths:
    workshop: /workspace/data/dpo_val.jsonl

dpo:
  max_num_epochs: 1
  max_num_steps: 20
  val_period: 10
  val_at_start: true
  val_at_end: true
  val_batches: 4
  val_global_batch_size: 4
  val_micro_batch_size: 1
  reference_policy_kl_penalty: 0.05
  preference_loss_weight: 1
  sft_loss_weight: 0
  preference_average_log_probs: false
  sft_average_log_probs: false
  seed: 42

checkpointing:
  enabled: true
  checkpoint_dir: /workspace/work/dpo_checkpoints
  save_period: 10
  keep_top_k: 2
  save_optimizer: true
  higher_is_better: false
  metric_name: "val:validation-workshop_loss"

logger:
  log_dir: /workspace/work/dpo_logs
  monitor_gpus: true
  num_val_samples_to_print: 1
  tensorboard_enabled: false
  wandb_enabled: false
  mlflow_enabled: false
  swanlab_enabled: false

cluster:
  num_nodes: 1
  gpus_per_node: 1
"""

(NB_DIR / "recipes" / "dpo_1gpu.yaml").write_text(DPO_RECIPE)
print("wrote recipes/dpo_1gpu.yaml | model =", MODEL_PATH)


wrote recipes/dpo_1gpu.yaml | model = Qwen/Qwen2.5-0.5B


## 3. Run DPO (`run_dpo.py` in the container)

Inside the container (production `run_dpo.sh` runs as root with the venv python):

```bash
cd /opt/nemo-rl
/opt/nemo_rl_venv/bin/python -u examples/run_dpo.py --config /workspace/recipes/dpo_1gpu.yaml
```

We pass `--entrypoint ""` so Docker does not invoke `nvidia_entrypoint.sh` (which
looks for `python` on PATH and exits 127).

Watch `val:validation-workshop_loss` trend down. Checkpoints land under
`work/dpo_checkpoints/`.


In [3]:
import sys

(WORK_DIR / "dpo_checkpoints").mkdir(parents=True, exist_ok=True)
(WORK_DIR / "dpo_logs").mkdir(parents=True, exist_ok=True)

proc = subprocess.Popen(
    DOCKER + [
        "--workdir", "/opt/nemo-rl", CONTAINER,
        NEMO_PYTHON, "-u", "examples/run_dpo.py",
        "--config", "/workspace/recipes/dpo_1gpu.yaml",
    ],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:
    sys.stdout.write(line)
rc = proc.wait()
print("\nDPO exited with code", rc)


/opt/nemo-rl/3rdparty/Megatron-LM-workspace/Megatron-LM/megatron/core/distributed/fsdp/src/megatron_fsdp/mixed_precision.py:115: UserWarning: Transformer Engine and Apex are not installed. Falling back to local implementations of multi_tensor_applier and multi_tensor_scale
  warnings.warn(
/opt/nemo-rl/3rdparty/Megatron-LM-workspace/Megatron-LM/megatron/core/optimizer/__init__.py:25: UserWarning: Transformer Engine and Apex are not installed. Falling back to Torch optimizers.
  warnings.warn(
/opt/nemo-rl/3rdparty/Megatron-LM-workspace/Megatron-LM/megatron/core/optimizer/optimizer.py:29: UserWarning: Transformer Engine and Apex are not installed. Falling back to local implementations of multi_tensor_applier and multi_tensor_scale
  warnings.warn(
/opt/nemo-rl/3rdparty/Megatron-LM-workspace/Megatron-LM/megatron/core/optimizer/clip_grads.py:32: UserWarning: Transformer Engine and Apex are not installed. Falling back to local implementations of multi_tensor_applier, multi_tensor_l2norm, a

## 4. Inspect the checkpoint

The best checkpoint (lowest `val:validation-data-dpo_val_loss`) under
`work/dpo_checkpoints/` is the DPO-aligned model for M8 evaluation.


In [4]:
ckpt_root = WORK_DIR / "dpo_checkpoints"
print("checkpoints:")
if ckpt_root.exists():
    for d in sorted(ckpt_root.glob("*")):
        print("  ", d.name)
else:
    print("  (none yet)")
saved = sorted(ckpt_root.glob("**/*")) if ckpt_root.exists() else []
print("\ncheckpoint dir:", ckpt_root)
print("Next: M8 evaluator compares base / SFT / DPO.")


checkpoints:
   step_9

checkpoint dir: /data/workshop/M7-model_training/work/dpo_checkpoints
Next: M8 evaluator compares base / SFT / DPO.
